# Notes
This is to extract SMILES from input files, and to calculate descriptors for them. The outputs will be saved at `input_data.json` under `<working dir>`

> - The default working dir is `Viability`
> - If want to use custom working dir, the corresponding parameters should also be modified in the following steps  


# Imports

In [ ]:
# Imports
import pandas as pd
import numpy as np
import os, sys
import json

from functools import reduce

# Input
Define your own inputs here

In [ ]:
# Load and define input DATA
df1 = pd.read_excel("Viability/data/食品添加剂all.xlsx")
df2 = pd.read_excel("Viability/data/中药all.xlsx")

DATA = pd.concat([
    df1, 
    df2,
], axis=0)

# Define SMILES column
SMILES_COL = "SMILES"


In [ ]:
# Extract SMILES
new_data_list = []
for idx,i in enumerate(DATA.itertuples()):
    d = dict(
        smiles=getattr(i, SMILES_COL, ""),
    )
    new_data_list.append(d)

new_data_df = pd.DataFrame(new_data_list)

print(f"Input shape: {new_data_df.shape}")

# Calculate descriptors
~1 min per 2000 SMILES

In [ ]:
# Descriptors
from Descriptors.mordred import get_rdkit_desc

def get_col(df:pd.DataFrame):
    'return names of descriptors'
    col = df.columns.drop("smiles", errors='ignore').to_list()
    return col

# Core RDKit descriptors
rdkit_desc = get_rdkit_desc(new_data_df['smiles'])
rdkit_col = get_col(rdkit_desc)


In [ ]:
# Merge descriptors and SMILES
def _merge(x,y):
    return pd.merge(x, y, on='smiles', how='inner')

def merge_recursively(df_list):
    return reduce(_merge, df_list)

df_list = [
    new_data_df,
    rdkit_desc,
]

final_data_df = merge_recursively(df_list)

print(f"Dataset: {len(final_data_df)}\nDescriptors: {len(get_col(final_data_df))}")

# Write outputs
Remember to create a dir for first use

In [ ]:
# Write outputs

final_data_df.to_json("Viability/data_input.json")